### Cell 1: Setup

In [1]:
import sys; sys.path.insert(0, "..")
import json
from src.agents.graph import run_query, build_graph
from src.agents.state import AgentState
print("imports ok")
# Sanity-check graph topology
app = build_graph()
print("nodes:", list(app.get_graph().nodes.keys()))

c:\Users\bhush\DevEnvironments\global_env\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


imports ok
nodes: ['__start__', 'source', 'political', 'military', 'critique', 'narrative', '__end__']


---
---
### Cell 2: Run a query

In [2]:
QUERY = "Who commanded the INA at the Red Fort trials in 1945 and what were the charges?"
print(f"Running: {QUERY!r}")
state = run_query(QUERY)
print(f"Error:        {state.get('error')}")
print(f"Critique loops: {state.get('critique_loop_count')}")
print(f"Critique passed: {state.get('critique_passed')}")

Running: 'Who commanded the INA at the Red Fort trials in 1945 and what were the charges?'


{"timestamp": "2026-07-02T05:30:06.834108+00:00", "level": "INFO", "name": "src.storage.db_client", "message": "Query executed successfully"}
{"timestamp": "2026-07-02T05:30:07.537176+00:00", "level": "INFO", "name": "src.retrieval.bm25_retriever", "message": "BM25 index built"}
c:\Users\bhush\DevEnvironments\global_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
{"timestamp": "2026-07-02T05:31:17.296065+00:00", "level": "INFO", "name": "src.processing.embedder", "message": "Loading embedding model"}
{"timestamp": "2026-07-02T05:31:27.394680+00:00", "level": "INFO", "name": "src.processing.embedder", "message": "Embedding model loaded"}
{"timestamp": "2026-07-02T05:31:30.674532+00:00", "level": "INFO", "name": "src.storage.db_client", "message": "Query executed successfully"}
{"timestamp": "2026-07-02T05:31:30

Error:        None
Critique loops: 3
Critique passed: True


### Cell 3 — Inspect source output

In [3]:
so = state.get("source_output")
assert so is not None, "source_output is None — check source_agent logs"
chunks = json.loads(so["content"])
print(f"Chunks retrieved: {len(chunks)}")
print(f"Confidence: {so['confidence']}")
print(f"Citations: {so['citations'][:5]}")
print("\nTop 3 chunks:")
for c in chunks[:3]:
    print(f"  [{c['bias_tag']:20s}] {c['doc_id']} | {c['text'][:120]}\n")

Chunks retrieved: 20
Confidence: 1.0
Citations: ['ia_indian_national_army_20180101_008', 'ia_indian_national_army_20201107_005', 'ia_indian_national_army_20170101_009', 'ia_indian_national_army_20180101_008', 'ia_indian_national_army_20180101_008']

Top 3 chunks:
  [academic            ] ia_indian_national_army_20180101_008 | to work for IIL or INA. The membership of the IIL peaked at 350,000. Over a lakh local Indians in South-east Asia volunt

  [academic            ] ia_indian_national_army_20201107_005 | allegations against the INA like using force on the prisoners of the war to join the Azad Hind Fauj, killing unwilling s

  [academic            ] ia_indian_national_army_20170101_009 | “Give me blood! I will give you freedom.” Bose's magic appeal worked wonders with even the civilian population: barriste



### Cell 4 — Inspect political output

In [4]:
po = state.get("political_output")
assert po is not None, "political_output is None — check political_agent logs"
print("=== POLITICAL ANALYSIS ===")
print(po["content"])
print(f"\nCitations ({len(po['citations'])}):", po["citations"][:5])

=== POLITICAL ANALYSIS ===
**BENEFICIARY**  
The framing of the evidence primarily benefits the Indian National Army (INA) and its leaders, particularly Netaji Subhas Chandra Bose. The portrayal of the INA as a heroic force fighting for India's freedom, despite the allegations of coercion and violence, serves to elevate their status in the narrative of India's struggle for independence. Additionally, the framing benefits nationalist sentiments among the Indian populace, as it emphasizes collective action against colonial rule.

**OMISSIONS**  
The evidence omits a nuanced discussion of the complexities surrounding the INA's actions, particularly the allegations of coercion and violence against prisoners and unwilling soldiers. It also overlooks the perspectives of those who opposed the INA, including segments of the Indian populace and British officials who viewed the INA's methods as treacherous. Furthermore, the broader context of the socio-political environment in India during this 

### Cell 5 — Inspect military output

In [5]:
mo = state.get("military_output")
assert mo is not None, "military_output is None — check military_agent logs"
print("=== MILITARY ANALYSIS ===")
print(mo["content"])
print(f"\nCitations ({len(mo['citations'])}):", mo["citations"][:5])

=== MILITARY ANALYSIS ===
### PLAUSIBLE
- The claim that the Indian National Army (INA) reached a force of 50,000 is plausible given the context of World War II and the recruitment efforts in Southeast Asia. The logistical possibility of mobilizing such numbers, especially with the support of local populations, aligns with the historical context.
- The mention of various commanders such as MZ Kiani, Inayat Kiani, Gulzara Singh, Gurubaksh Singh Dhillon, and Shah Nawaz Khan is consistent with historical records of the INA's leadership structure during the trials at the Red Fort in 1945.

### IMPLAUSIBLE
- The allegations of the INA using force on prisoners of war to join their ranks, including killing unwilling soldiers, while documented, may be exaggerated in the context of the overall narrative of the INA's struggle. While coercion did occur, the scale and nature of these actions may not align with the broader military conduct expected during that period.
- The claim that the INA "snat

### Cell 6 — Inspect critique output

In [6]:
co = state.get("critique_output")
assert co is not None, "critique_output is None — check critique_agent logs"
print("=== CRITIQUE OUTPUT ===")
print(co["content"])
try:
    parsed = json.loads(co["content"])
    print(f"\nConfidence: {parsed.get('confidence')}")
    print(f"Decision:   {parsed.get('decision')}")
    print(f"Contradictions ({len(parsed.get('contradictions', []))}):")
    for c in parsed.get("contradictions", []):
        print(f"  - {c}")
except json.JSONDecodeError:
    print("(critique content is not JSON — model deviated from prompt)")

=== CRITIQUE OUTPUT ===
{"contradictions": ["The first output lists several commanders of the INA but does not specify who commanded during the Red Fort trials, while the military output confirms the leadership structure but does not clarify the specific command during the trials. Additionally, the military output mentions the trials but does not detail the charges against the INA members, which is a critical aspect of the query."], "confidence": 0.4, "decision": "LOOP", "critique_notes": "The outputs provide some information about the INA's leadership and context but fail to directly address the specific command during the Red Fort trials and the charges against the INA members, leading to a material contradiction that requires resolution."}

Confidence: 0.4
Decision:   LOOP
Contradictions (1):
  - The first output lists several commanders of the INA but does not specify who commanded during the Red Fort trials, while the military output confirms the leadership structure but does not 

### Cell 7 — Narrative output

In [7]:
no = state.get("narrative_output")
assert no is not None, "narrative_output is None — graph did not reach narrative node"
print("=== FINAL NARRATIVE ===")
print(no["content"])
print(f"\nConfidence:  {no['confidence']}")
print(f"Citations:   {no['citations']}")

=== FINAL NARRATIVE ===
## Political Reality
The Red Fort trials in 1945 were a significant moment in India's struggle for independence, as they centered around the Indian National Army (INA) and its leaders. The primary commander of the INA during the trials was Major General Shah Nawaz Khan, who, along with other leaders like Colonel Gurbaksh Singh Dhillon and Colonel Prem Kumar Sahgal, faced charges of waging war against the King-Emperor and conspiracy to overthrow the British government in India [ia_indian_national_army_20220615_004]. The political framing of these trials served to galvanize nationalist sentiments and position the INA as a symbol of resistance against colonial rule. The trials were not merely legal proceedings; they were a platform for the INA to articulate its vision of an independent India, thus benefiting nationalist narratives and bolstering the image of its leaders as freedom fighters [ia_indian_national_army_20170101_009].

## Military Reality
From a military

### Cell 8 — Debug log

In [8]:
print("=== DEBUG LOG ===")
for line in state.get("debug_log", []):
    print(" ", line)

=== DEBUG LOG ===
  source_agent: retrieved=20, kept=20, query_chars=79, duration_ms=124749 [triage bypassed]
  military_agent: chunks=20, duration_ms=5435
  political_agent: chunks=20, bias_tags=['academic'], duration_ms=5901
  critique_agent: loop=1, decision=LOOP, confidence=0.4, contradictions=1, duration_ms=3606
  political_agent: chunks=20, bias_tags=['academic'], duration_ms=4546
  critique_agent: loop=2, decision=LOOP, confidence=0.4, contradictions=1, duration_ms=4395
  political_agent: chunks=20, bias_tags=['academic'], duration_ms=5365
  critique_agent: forced PASS at loop limit
  narrative_agent: response_chars=3596, citations=5, duration_ms=12890


### Cell 9 — Quality gate

In [9]:
# Run cells 1–8 first. Then answer each question below.
#
# 1. Does source_output contain chunks relevant to the query? (Cell 3)  -- Yes
# 2. Does political_output correctly identify at least one BENEFICIARY and one OMISSION? (Cell 4) -- Yes
# 3. Does military_output produce all three sections PLAUSIBLE/IMPLAUSIBLE/UNCERTAIN? (Cell 5) -- Yes
# 4. Does critique_output parse as valid JSON with a confidence score? (Cell 6) -- Yes
# 5. Does the narrative contain all four required section headers? (Cell 7) -- Yes
# 6. Is the narrative factually grounded (cites specific doc_ids, not vague)? (Cell 7) -- Yes
# 7. Is critique_loop_count <= 2? (Cell 2 — if it hit 3, investigate why) --it hit 3, reason is confidence of 0.4, maybe less data
#
# Common fixes:
#   Source quality low     → re-run Week 2 embedder, check chunker output
#   Political/military off → adjust SYSTEM prompt temperature in agent file
#   Critique always LOOPs  → lower critique temperature, check JSON parse logic
#   Narrative misses headers → add stop_sequences to call_hf in narrative_agent
print("Fill in quality assessment above to check working")

Fill in quality assessment above to check working
